In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 8
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'dejavusans'

In [2]:
import sys
from pathlib import Path


In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import figurefirst as fifi
import pynumdiff
from braid_analysis.flymath import wrap_angle
from splitflow import set_zorder_functions
import numpy as np

In [4]:
df = pd.read_parquet('unifying_sim_data.parquet')

In [5]:
option = 'saccadic' # 'saccadic' 'original'
TAIL = 200

dt = 0.01

if option == 'saccadic':
    x_label = 'x_sac'
    y_label = 'y_sac'
    course_label = 'course_sac'
    course_x_label = 'course_x_sac'
    course_y_label = 'course_y_sac'
    angvel_label = 'angvel_sac'
elif option == 'smooth':
    df['course_x_smooth'], _ = pynumdiff.savgoldiff(df.course_x.values, dt, [3, 101, 101])
    df['course_y_smooth'], _ = pynumdiff.savgoldiff(df.course_y.values, dt, [3, 101, 101])
    df['course_smooth'] = np.arctan2(df['course_y_smooth'], df['course_x_smooth'])

    speed = 0.3
    vx = df['course_x_smooth']*speed
    vy = df['course_y_smooth']*speed
    
    x = np.cumsum(vx)*dt
    y = np.cumsum(vy)*dt

    df['x_smooth'] = x
    df['y_smooth'] = y

    x_label = 'x_smooth'
    y_label = 'y_smooth'
    course_label = 'course_smooth'
    course_x_label = 'course_x_smooth'
    course_y_label = 'course_y_smooth'
    angvel_label = 'angvel_smooth'
    
elif option == 'original':
    x_label = 'x'
    y_label = 'y'
    course_label = 'course'
    course_x_label = 'course_x'
    course_y_label = 'course_y'
    angvel_label = 'angvel'

In [6]:
def plot_single_bar(ax, height, color, width=0.6, bottom=0, **kwargs):
    """
    Plot a single bar with no edge/outline.

    Parameters
    ----------
    ax      : matplotlib Axes
    height  : height of the bar
    color   : fill color (any matplotlib color spec)
    width   : bar width (default 0.6)
    bottom  : bar base y-value (default 0, useful for stacking)
    **kwargs: forwarded to ax.bar()

    Returns
    -------
    bar : BarContainer
    """
    bar = ax.bar(
        0, height,
        width=width,
        bottom=bottom,
        color=color,
        edgecolor="none",
        linewidth=0,
        **kwargs,
    )
    return bar

In [7]:
def remove_jumps(arr, threshold):
    """
    Replace values with np.nan where the signal jumps by more than
    `threshold` relative to the previous sample.

    Parameters
    ----------
    arr       : 1D numpy array
    threshold : scalar — jumps larger than this (in absolute value) are removed

    Returns
    -------
    out : copy of arr with jump locations set to np.nan
    """
    out = arr.astype(float).copy()
    jumps = np.abs(np.diff(out)) > threshold
    out[1:][jumps] = np.nan
    return out

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from typing import List, Optional, Tuple


def circular_distribution_plot(
    directions: List[float],
    magnitudes: List[float],
    ax: Optional[plt.Axes] = None,
    bins: int = 36,
    radial_limit: Optional[float] = None,
    log_scale: bool = False,
    normalize: bool = False,
    figsize: Tuple[int, int] = (7, 7),
) -> plt.Axes:
    """
    Create a circular distribution (polar) plot for vector directions and magnitudes.

    Bar height encodes mean magnitude per bin. Bar darkness encodes count density
    (how many vectors fall in that bin), mapped from light gray (few) to black (many).
    Together these show both the strength and prevalence of vectors in each direction.

    Parameters
    ----------
    directions   : Sequence of direction angles in radians (-pi to pi).
    magnitudes   : Corresponding magnitudes for each direction vector.
    ax           : Existing polar Axes to draw into. If None, a new figure is created.
    bins         : Number of angular bins around the circle (default 36 -> 10 deg each).
    radial_limit : Upper limit of the radial axis. Applied after any transforms.
    log_scale    : If True, apply log1p transform to mean magnitudes before plotting.
    normalize    : If True, normalize mean magnitudes to sum to 1 across bins.
                   Useful for comparing distributions with different overall scales.
    figsize      : Figure size in inches, used only when ax is None.

    Returns
    -------
    ax           : The polar Axes object.

    Example
    -------
    >>> import numpy as np
    >>> rng = np.random.default_rng(42)
    >>> dirs = rng.uniform(-np.pi, np.pi, 200)
    >>> mags = rng.rayleigh(scale=5, size=200)
    >>> ax = circular_distribution_plot(dirs, mags)
    """
    directions = np.asarray(directions, dtype=float)
    magnitudes = np.asarray(magnitudes, dtype=float)

    if directions.shape != magnitudes.shape:
        raise ValueError("directions and magnitudes must have the same length.")

    # --- Binning ---
    radians = directions % (2 * np.pi)
    bin_edges = np.linspace(0, 2 * np.pi, bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]

    bin_indices = np.clip(np.digitize(radians, bin_edges) - 1, 0, bins - 1)

    bin_total_mag = np.zeros(bins)
    bin_counts = np.zeros(bins, dtype=float)
    for idx, mag in zip(bin_indices, magnitudes):
        bin_total_mag[idx] += mag
        bin_counts[idx] += 1

    # Mean magnitude per bin (bar height); empty bins stay at 0
    bin_mean_mag = np.where(bin_counts > 0, bin_total_mag / bin_counts, 0.0)

    # --- Transforms ---
    if normalize and bin_mean_mag.sum() > 0:
        bin_mean_mag = bin_mean_mag / bin_mean_mag.sum()

    if log_scale:
        bin_mean_mag = np.log1p(bin_mean_mag)

    # --- Bar colours: count density mapped to gray (light = few, dark = many) ---
    count_norm = Normalize(vmin=0, vmax=bin_counts.max() if bin_counts.max() > 0 else 1)
    # Map normalised count to a gray value: 0 -> 0.85 (light), 1 -> 0.15 (dark)
    gray_values = 0.85 - 0.70 * count_norm(bin_counts)
    bar_colors = [(g, g, g, 1.0) for g in gray_values]

    # --- Axes ---
    if ax is None:
        _, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"})

    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

    # --- Bars ---
    bars = ax.bar(
        bin_edges[:-1],
        bin_mean_mag,
        width=bin_width,
        color=bar_colors,
        edgecolor="none",
        linewidth=0.4,
        align="edge",
    )

    # --- Clean up labels and ticks ---
    ax.set_xticklabels([])
    #ax.set_yticklabels([])
    ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.4)

    if radial_limit is not None:
        ax.set_ylim(0, radial_limit)

    ax.spines["polar"].set_visible(False)
    
    return ax



In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_polar_arrow(ax, direction_rad, length, thickness=2, color='black', alpha=1.0):
    """
    Draw an arrow on a polar axis.

    Parameters
    ----------
    ax          : matplotlib polar Axes
    direction_rad : float — arrow direction in radians (0 = right, pi/2 = up)
    length      : float — arrow length in data units (same as radial axis)
    thickness   : float — line/arrow width in points (default 2)
    color       : str or color — arrow color
    alpha       : float — opacity (0–1)
    """
    ax.annotate(
        "",
        xy=(direction_rad, length),        # arrowhead (tip)
        xytext=(direction_rad, 0),         # arrow base (origin)
        arrowprops=dict(
            arrowstyle="-|>",
            color=color,
            lw=thickness,
            mutation_scale=thickness * 4,  # scales the arrowhead proportionally
        ),
        alpha=alpha,
    )

In [10]:
"""
animate_fifi.py
---------------
Frame-by-frame animation using figurefirst + ghost_tail helpers.

Pipeline per frame
------------------
  1. Clear & redraw all matplotlib artists on the axes
  2. Embed the updated mpl figure into the SVG layout via figurefirst
  3. Write the composite SVG to disk
  4. Rasterise that SVG to a PNG via cairosvg

Why a plain for-loop instead of FuncAnimation?
  figurefirst's write cycle (append_figure_to_layer → write_svg) is
  file-based and has no concept of a live display loop.  FuncAnimation
  is designed for interactive/video rendering — it calls blit or draw
  but never writes files.  A for-loop gives you full control and makes
  the SVG→PNG step trivial to slot in.

Dependencies
------------
  pip install figurefirst cairosvg matplotlib numpy
"""

import numpy as np
import matplotlib.pyplot as plt
import figurefirst as fifi
import cairosvg
from pathlib import Path

from splitflow.ghost_tail import (
    setup_xy_ghost_plot,
    setup_time_ghost_plot,
    update_ghost_tail,
    update_current_point,
)

# ── Output directory ──────────────────────────────────────────────────────────
OUT = Path(option+"_frames")
OUT.mkdir(exist_ok=True)

SVG_TEMPLATE = "animation_figure.svg"   # your figurefirst layout file

def build_layout():
    """
    (Re-)construct the figurefirst layout and return axes handles.

    figurefirst embeds mpl figures directly into the SVG document tree,
    so the layout object must be re-created each frame — it is not
    cheaply reusable across write cycles.
    """
    layout = fifi.svg_to_axes.FigureLayout(
        SVG_TEMPLATE,
        autogenlayers=True,
        make_mplfigures=True,
        hide_layers=[],
    )
    plt.close("all")   # figurefirst opens figures internally; close stale ones

    ax_trajec = layout.axes[('animation', 'trajec')]
    ax_wind_hist = layout.axes[('animation', 'wind_hist')]
    
    ax_wind_speed = layout.axes[('animation', 'wind_speed')]
    ax_wind_direction = layout.axes[('animation', 'wind_direction')]
    ax_course = layout.axes[('animation', 'course')]
    
    ax_affine = layout.axes[('animation', 'affine')]

    ax_slope = None # layout.axes[('animation', 'slope')]
    ax_wind_vecstr = layout.axes[('animation', 'wind_vecstr')]
    ax_wind_direction_bar = layout.axes[('animation', 'wind_direction_bar')]
    ax_circle = layout.axes[('animation', 'circle')]
    
    ax_goal = layout.axes[('animation', 'goal')]
    ax_scale = layout.axes[('animation', 'scale')]
    ax_axis_ratio = layout.axes[('animation', 'axis_ratio')]
    
    return layout, ax_trajec, ax_wind_speed, ax_wind_direction, ax_circle, ax_affine, ax_course, ax_axis_ratio, ax_slope, ax_wind_hist, ax_wind_vecstr, ax_goal, ax_scale, ax_axis_ratio, ax_wind_direction_bar


def render_frame(frame_idx: int, stride: int = 1):
    """
    Render a single frame.

    Parameters
    ----------
    frame_idx : 0-based index into the data arrays
    stride    : only write every nth frame (default = every frame)

    Returns
    -------
    svg_path : Path to the written SVG
    png_path : Path to the written PNG
    """
    if frame_idx % stride != 0:
        return None, None

    # ── 1. Build fresh layout + axes ─────────────────────────────────────────
    layout, ax_trajec, ax_wind_speed, ax_wind_direction, ax_circle, ax_affine, ax_course, ax_axis_ratio, ax_slope, ax_wind_hist, ax_wind_vecstr, ax_goal, ax_scale, ax_axis_ratio, ax_wind_direction_bar = build_layout()

    # ── 2. Draw ─────────────────────────────────

    # -------------------------------------------------------------------------
    # Trajectory
    x = df[x_label].values
    y = df[y_label].values
    lc_xy, pt_xy = setup_xy_ghost_plot(
        ax_trajec, x, y,
        tail_length=TAIL,
        tail_kw=dict(color="black", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="black", markersize=7),
    )
    ax_trajec.set_xlim(-.01, 1.2)
    ax_trajec.set_ylim(-0.1, 0.2)
    ax_trajec.set_aspect("equal")
    fifi.mpl_functions.adjust_spines(ax_trajec, [])
    
    update_ghost_tail(lc_xy, x, y, frame_idx)
    update_current_point(pt_xy, x, y, frame_idx)

    # wind histogram plot
    last = frame_idx
    first = np.max([0, last-100])
    _ = circular_distribution_plot(df.wind_direction.values[first:last], 
                                    df.wind_speed.values[first:last], 
                                    bins=50,
                                    ax=ax_wind_hist,
                                    normalize=False, radial_limit=0.5)
    ax_wind_hist.set_yticks([0, 0.25, 0.5])
    ax_wind_hist.set_yticklabels([])
    draw_polar_arrow(ax_wind_hist, 
                     df.wind_direction.values[last], 
                     0.5, #df.wind_speed.values[last], 
                     thickness=2, color='green', alpha=1.0)

    # -------------------------------------------------------------------------


    
    # -------------------------------------------------------------------------
    # Inputs
    
    # Slope
    # height = df['omega'].values[frame_idx]
    # color = 'black'
    # plot_single_bar(ax_slope, height, color, width=0.6, bottom=0)
    # ax_slope.set_xlim(-0.5, 0.5)
    # ax_slope.set_ylim(0, 8)
    # ax_slope.set_yticks([0, 2, 4, 6, 8])
    # fifi.mpl_functions.adjust_spines(ax_slope, ['left'],
    #                                  spine_locations={'left': 5, 'bottom': 5},
    #                                  tick_length=2.5,)
    #ax_slope.set_ylabel('(rad/s)')

    # Circle
    x = df['circular_basis_x'].values
    y = df['circular_basis_y'].values
    lc_xy, pt_xy = setup_xy_ghost_plot(
        ax_circle, x, y,
        tail_length=TAIL,
        tail_kw=dict(color="blue", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="blue", markersize=7),
    )
    ax_circle.set_xlim(-1.15, 1.15)
    ax_circle.set_ylim(-1.15, 1.15)
    ax_circle.set_aspect("equal")
    fifi.mpl_functions.adjust_spines(ax_circle, [])
    update_ghost_tail(lc_xy, x, y, frame_idx)
    update_current_point(pt_xy, x, y, frame_idx)

    # Wind vec str
    height = df['wind_vec_str'].values[frame_idx]
    color = 'black'
    plot_single_bar(ax_wind_vecstr, height, color, width=0.6, bottom=0)
    ax_wind_vecstr.set_xlim(-0.5, 0.5)
    ax_wind_vecstr.set_ylim(0, 1)
    ax_wind_vecstr.set_yticks([0, 0.5, 1])
    fifi.mpl_functions.adjust_spines(ax_wind_vecstr, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)
    #ax_wind_vecstr.set_ylabel('(rad/s)')

    # Wind direction
    #height = df['wind_direction'].values[frame_idx]
    color = 'black'
    height = wrap_angle(df['wind_direction'].values[frame_idx] + np.pi)
    ax_wind_direction_bar.plot(0, height, 'o', markerfacecolor='green', markeredgecolor='green', markersize=5)
    ax_wind_direction_bar.set_xlim(-0.5, 0.5)
    #ax_wind_direction_bar.set_ylim(-np.pi, np.pi)
    #ax_wind_direction_bar.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    #ax_wind_direction_bar.set_yticklabels(['$-\pi$', '$-\pi/2$', '$0$', '$\pi/2$', '$\pi$'])

    ax_wind_direction_bar.set_ylim(-np.pi, np.pi)
    ax_wind_direction_bar.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax_wind_direction_bar.set_yticklabels(['$0$', '$\pi/2$', '$\pi$', '$3\pi/2$', '$2\pi$'])
    
    fifi.mpl_functions.adjust_spines(ax_wind_direction_bar, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)
    #ax_wind_direction_bar.set_ylabel('(rad)')
    # -------------------------------------------------------------------------



    # -------------------------------------------------------------------------
    # Transform parameters
    # Goal
    height = df['goal'].values[frame_idx]
    if df['behavior'][frame_idx] == 'explore':
        color = 'brown'
    else:
        color = 'green'
    ax_goal.plot(0, height, 'o', markerfacecolor=color, markeredgecolor=color, markersize=5)
    ax_goal.set_xlim(-0.5, 0.5)
    ax_goal.set_ylim(-np.pi, np.pi)
    ax_goal.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax_goal.set_yticklabels(['$-\pi$', '', '$0$', '', '$\pi$'])
    fifi.mpl_functions.adjust_spines(ax_goal, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)
    #ax_goal.set_ylabel('(rad)')

    # Scale
    height = df['scale'].values[frame_idx]
    color = 'black'
    plot_single_bar(ax_scale, height, color, width=0.6, bottom=0)
    ax_scale.set_xlim(-0.5, 0.5)
    ax_scale.set_ylim(0, 1)
    ax_scale.set_yticks([0, 0.5, 1])
    fifi.mpl_functions.adjust_spines(ax_scale, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)
    #ax_scale.set_ylabel('(rad)')

    # Axis ratio
    height = df['lambda'].values[frame_idx]
    color = 'black'
    plot_single_bar(ax_axis_ratio, height, color, width=0.6, bottom=0)
    ax_axis_ratio.set_xlim(-0.5, 0.5)
    ax_axis_ratio.set_ylim(0, 1)
    ax_axis_ratio.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    fifi.mpl_functions.adjust_spines(ax_axis_ratio, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)
    #ax_axis_ratio.set_ylabel('(rad)')
    # -------------------------------------------------------------------------
    
    # -------------------------------------------------------------------------
    # Output
    x = df[course_x_label].values
    y = df[course_y_label].values
    lc_xy, pt_xy = setup_xy_ghost_plot(
        ax_affine, x, y,
        tail_length=TAIL,
        tail_kw=dict(color="magenta", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="magenta", markersize=7),
    )
    ax_affine.set_xlim(-1.1, 1.1)
    ax_affine.set_ylim(-1.1, 1.1)
    ax_affine.set_aspect("equal")
    fifi.mpl_functions.adjust_spines(ax_affine, [])

    update_ghost_tail(lc_xy, x, y, frame_idx)
    update_current_point(pt_xy, x, y, frame_idx)


    # -------------------------------------------------------------------------
    # Time-series plot
    t = df['time'].values
    v = df['wind_speed'].values
    lc_ts, pt_ts = setup_time_ghost_plot(
        ax_wind_speed, t, v,
        tail_length=TAIL,
        tail_kw=dict(color="black", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="black", markersize=7),
    )
    ax_wind_speed.set_xlim(t[0], t[-1])
    ax_wind_speed.set_ylim(0, 1)
    ax_wind_speed.set_ylabel('Wind speed (m/s)')
    ax_wind_speed.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    fifi.mpl_functions.adjust_spines(ax_wind_speed, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)

    update_ghost_tail(lc_ts, t, v, frame_idx)
    update_current_point(pt_ts, t, v, frame_idx)

    # -------------------------------------------------------------------------
    # Time-series plot
    t = df['time'].values
    v = wrap_angle(df['wind_direction'].values + np.pi)
    v = remove_jumps(v, 3)
    lc_ts, pt_ts = setup_time_ghost_plot(
        ax_wind_direction, t, v,
        tail_length=TAIL,
        tail_kw=dict(color="black", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="black", markersize=7),
    )
    ax_wind_direction.set_xlim(t[0], t[-1])
    ax_wind_direction.set_ylim(-np.pi, np.pi)
    ax_wind_direction.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax_wind_direction.set_yticklabels(['$0$', '$\pi/2$', '$\pi$', '$3\pi/2$', '$2\pi$'])
    ax_wind_direction.set_ylabel('Wind dir. (rad)')
    fifi.mpl_functions.adjust_spines(ax_wind_direction, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)

    update_ghost_tail(lc_ts, t, v, frame_idx)
    update_current_point(pt_ts, t, v, frame_idx)

    # -------------------------------------------------------------------------
    # Time-series plot
    t = df['time'].values
    v = df[course_label].values
    v = remove_jumps(v, 3)
    lc_ts, pt_ts = setup_time_ghost_plot(
        ax_course, t, v,
        tail_length=TAIL,
        tail_kw=dict(color="magenta", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="magenta", markersize=7),
    )
    ax_course.set_xlim(0, 30)
    ax_course.set_ylim(-np.pi, np.pi)
    ax_course.set_ylabel('Course (rad)')
    ax_course.set_xlabel('Time (s)')
    ax_course.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax_course.set_yticklabels(['$-\pi$', '$-\pi/2$', '$0$', '$\pi/2$', '$\pi$'])
    ax_course.set_xticks([0, 5, 10, 15, 20, 25, 30])
    fifi.mpl_functions.adjust_spines(ax_course, ['left', 'bottom'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)

    update_ghost_tail(lc_ts, t, v, frame_idx)
    update_current_point(pt_ts, t, v, frame_idx)

    # -------------------------------------------------------------------------
    


    # -------------------------------------------------------------------------
    # svg items
    item = layout.svgitems['text_wind_type']
    if df['wind_type'][frame_idx] != 'still':
        item.text = df['wind_type'][frame_idx] + ' wind'
    else:
        item.text = df['wind_type'][frame_idx] + ' air'

    item = layout.svgitems['text_behavior']
    item.text = 'Behavior: ' + df['behavior'][frame_idx]
    if df['behavior'][frame_idx] == 'explore':
        item.style['fill'] = '#964B00'
    else:
        item.style['fill'] = '#008000'
    

    item = layout.svgitems['text_behavior_input']
    if df['behavior'][frame_idx] == 'explore':
        item.text = 'arbitrary'
        item.style['fill'] = '#964B00'
    else:
        item.text = 'upwind'
        item.style['fill'] = '#008000'

    item = layout.svgitems['text_searching']
    if df['behavior'][frame_idx] == 'surge' or df['behavior'][frame_idx] == 'explore':
        item.style['fill-opacity'] = 0.
    else:
        item.style['fill-opacity'] = 1.

    item = layout.svgitems['text_goal_oriented']
    if df['behavior'][frame_idx] == 'surge' or df['behavior'][frame_idx] == 'explore':
        item.style['fill-opacity'] = 1.
    else:
        item.style['fill-opacity'] = 0.

    
    item = layout.svgitems['text_laminar']
    item.style['fill-opacity'] = 0.2
    
    item = layout.svgitems['text_unsteady']
    item.style['fill-opacity'] = 0.2

    
    item = layout.svgitems['text_low']
    item.style['fill-opacity'] = 0.2

    
    item = layout.svgitems['text_still']
    item.style['fill-opacity'] = 0.2

    current_wind = df['wind_type'][frame_idx]
    item = layout.svgitems['text_'+current_wind]
    item.style['fill-opacity'] = 1

    layout.apply_svg_attrs()


    
    # RASTERIZE

    #axes = [ax_trajec, ax_wind_speed, ax_wind_direction, ax_circle, ax_affine, ax_course, ax_axis_ratio, ax_slope, ax_wind_hist, ax_wind_vecstr, ax_goal, ax_scale, ax_axis_ratio, ax_wind_direction_bar]
    #for ax in axes:
    #    ax.set_rasterization_zorder(-1000)

    # ── 3. Embed mpl figure into the SVG layout ───────────────────────────────
    layout.append_figure_to_layer(
        layout.figures["animation"], "animation", cleartarget=True
    )

    svg_path = OUT / f"frame_{frame_idx:04d}.svg"
    layout.write_svg(str(svg_path))

    # ── 4. Rasterise SVG → PNG ────────────────────────────────────────────────
    png_path = svg_path.with_suffix(".png")
    cairosvg.svg2png(url=str(svg_path), write_to=str(png_path), dpi=150)

    # Delete svg
    os.remove(svg_path)

    plt.close("all")   # prevent figure accumulation
    return svg_path, png_path


# ── Main loop ─────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    STRIDE = 2   # render every 4th frame; set to 1 for every frame
    N = df.shape[0]

    for i in range(0, N):
        svg_path, png_path = render_frame(i, stride=STRIDE)
        if png_path:
            print(f"  frame {i:04d} → {png_path}")

    print(f"\nDone. Frames written to {OUT.resolve()}/")
    print("Combine into a video with e.g.:")
    print("  ffmpeg -r 30 -pattern_type glob -i 'saccadic_frames/*.png' -c:v libx264 -pix_fmt yuv420p -vf 'crop=trunc(iw/2)*2:trunc(ih/2)*2' out_saccadic.mp4")

/var/folders/63/jqdzyjdd4bnghtd7s779bx500000gp/T/ipykernel_9710/2062819834.py:69: RuntimeWarning: invalid value encountered in divide
  bin_mean_mag = np.where(bin_counts > 0, bin_total_mag / bin_counts, 0.0)


RuntimeError: Failed to process string with tex because latex could not be found